# 05 — Hyperparameter Tuning

Take the winning model + winning feature set and tune. Optuna is the modern default for this — Bayesian search beats grid search with the same compute budget.

If you want grid search instead: `from sklearn.model_selection import GridSearchCV`

In [ ]:
# pip install optuna  # if not already installed
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

## Load + use the winning feature set from 04

In [ ]:
# TODO: copy the data-prep + feature-set code from 04 so this is reproducible
# X, y = ...
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Search space

For LightGBM the high-impact hyperparameters are:

- `num_leaves` — controls tree complexity (16–255)
- `learning_rate` — typically 0.01–0.1
- `n_estimators` paired with `learning_rate` (more trees, smaller rate)
- `min_child_samples` — leaf-size regularization (10–200)
- `reg_alpha`, `reg_lambda` — L1/L2 (0–1)
- `feature_fraction`, `bagging_fraction` — stochastic regularization (0.6–1.0)

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 1.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    # TODO: fit on a subset, score on a held-out validation fold.
    # For speed, use a single train/val split inside the objective rather than full CV.
    # model.fit(X_train_inner, y_train_inner, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(30)])
    # proba = model.predict_proba(X_val)[:, 1]
    # return roc_auc_score(y_val, proba)
    return 0.5  # placeholder

# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=30, show_progress_bar=True)
# print('Best AUC:', study.best_value)
# print('Best params:', study.best_params)

## Re-train final model with best params

In [ ]:
# TODO: fit final model on full train set with study.best_params,
# evaluate on the held-out test set, compare to v1's 0.714 AUC.
# Save as artifacts/model_v2.pkl + metrics_v2.json

## Bottom line

_AUC v1 (untuned): 0.714_

_AUC v2 (tuned): ___

_Worth shipping v2? If the gain is < 1pp it might not be._

_Best hyperparameters: ___